# Facial landmark tracking & size measurement from video

This notebook extracts the first frame of every `.avi` recording for a subject, lets you
manually click a fixed set of facial landmarks on each frame in **napari**, then computes
pairwise distances between landmarks (in pixels and, after calibration, in millimetres)
and saves plots + a CSV of raw coordinates.

**Who this is for:** if you're picking this up after the person who wrote it has left the
lab, read this cell and the notes above each section before running anything — the
landmark-clicking step in particular only works if you click points in a specific order
(explained below), and there is no automatic check that you clicked the *right* points,
only that you clicked the *right number* of points.

**Requirements:** a conda/python env with `napari`, `opencv-python` (`cv2`), `numpy`,
`pandas`, `matplotlib`. This notebook was built in an env called `move_deve`.

**What you'll need to edit before running:**
1. `subject_data_vid_path` — path to the folder of `.avi` files for the subject/session.
2. The frame numbering (`P{i+8}`) used when saving overlay images — see note below.
3. `REF_MM` — the true physical size (mm) of the calibration reference (currently
   assumed to be the distance between `headplate_a` and `headplate_p`).

**What it produces**, saved into `<subject_data_vid_path>/landmarks_outputs/`:
- `landmarks.csv` — raw x/y pixel coordinates of every landmark, one row per video/frame
  (saved one level up, directly in `subject_data_vid_path`)
- `P{n}.png` — each frame with landmarks overlaid, for visual QC
- `pairwise_distances_*.png` — distance-between-landmarks plots, in pixels and in mm


In [ ]:
import napari
import os
import numpy as np
import matplotlib.pyplot as plt

while os.path.basename(os.getcwd()) != 'photostim_deve':
    os.chdir('..')

from cv2 import VideoCapture

## 1. Point to the subject's video folder

Set `subject_data_vid_path` to the folder containing that subject's `.avi` files.
The cell above (imports) walks up the directory tree until it finds a folder named
`photostim_deve` and treats that as the project root — so run this notebook from
somewhere inside that project, and give the path below relative to that root.

In [ ]:
subject_data_vid_path = 'data_vid/jm064'

## 2. Find the video files

Lists every `.avi` in the folder above, sorted alphabetically. This sort order determines
frame order everywhere downstream (frame 0 = first file alphabetically, etc.) — rename
files first if you need a specific order.

In [ ]:
# now get all .avi filenames and sort them alphabetically
vid_filenames = [f for f in os.listdir(subject_data_vid_path) if f.endswith('.avi') and not f.startswith('.')]
vid_filenames.sort()
print(f'Found {len(vid_filenames)} .avi files in {subject_data_vid_path}:')
for vid_filename in vid_filenames:
    print(vid_filename)

## 3. Load the first frame of each video

Only the **first frame** of each `.avi` is used — this is a measurement of a static
landmark layout per recording, not a per-frame video analysis. If you need a different
frame (e.g. to skip an intro/blank frame), edit the `cap.read()` logic here.

In [ ]:
# now load the first frame of each file
frames = []
for vid_filename in vid_filenames:
    vid_path = os.path.join(subject_data_vid_path, vid_filename)
    # load the first frame of the video
    cap = VideoCapture(vid_path)
    ret, frame = cap.read()
    if ret:
        frames.append(frame)
    cap.release()

In [ ]:
frames = np.array(frames)


## 4. Sanity-check the frames

Quick visual check that every video loaded correctly and frames look as expected before
you spend time clicking landmarks on them.

In [ ]:
for i in range(len(frames)):
    plt.imshow(frames[i])
    plt.show()

## 5. Click landmarks in napari — READ BEFORE RUNNING

Running this cell opens a napari window with all frames loaded as a stack, plus an empty
`landmarks` points layer.

**You must click points in exactly this order** (this is not enforced by the code —
getting the order wrong will silently mislabel every landmark downstream):

1. Click **`headplate_a`** on every frame, in frame order (frame 0, 1, 2, ...)
2. Then click **`headplate_p`** on every frame
3. Then click **`snout`** on every frame
4. Then click **`ear_a`** on every frame
5. Then click **`ear_p`** on every frame
6. Then click **`eye`** on every frame
7. Then click **`mouth`** on every frame

That's `len(LANDMARKS) x n_frames` points total, added landmark-by-landmark (not
frame-by-frame). The code below groups your clicks into blocks of `n_frames` and assigns
landmark names by block index — so a missed or extra click on any frame shifts every
landmark after it.

To navigate between frames in napari, use the slider/arrows at the bottom of the viewer
(or `Ctrl`/`Cmd` + arrow keys). Zoom with scroll, pan by dragging.

Close the napari window (or stop it) once all points are placed — the notebook execution
resumes after `napari.run()` returns.

In [ ]:
LANDMARKS = ["headplate_a", "headplate_p", "snout", "ear_a", "ear_p", "eye", "mouth"]

stack = np.stack(frames)          # (6, H, W, 3) from your cv2 frames
viewer = napari.Viewer()
viewer.add_image(stack, rgb=True, name="frames")

points = viewer.add_points(
    ndim=3,          # (frame, y, x)
    name="landmarks",
    size=8,
    face_color="cyan",
)

napari.run()

## 6. Turn clicks into a DataFrame

Reshapes the raw click coordinates into one row per frame with `<landmark>_x` /
`<landmark>_y` columns, using the click order described above.

**If this cell raises an `AssertionError`**, the number of points you placed doesn't match
`n_frames x n_landmarks`. Go back to the napari window (re-run the cell above, or reopen
the `viewer`/`points` objects if still in memory) and check for missed or duplicate
clicks before continuing — the count check catches *how many* points there are, not
whether they're in the right order, so also eyeball the overlay plots in step 8 before
trusting the numbers.

In [ ]:
import pandas as pd

n_frames = stack.shape[0]
n_landmarks = len(LANDMARKS)
data = points.data  # (N, 3): frame, y, x, in click order

assert len(data) == n_frames * n_landmarks, (
    f"expected {n_frames * n_landmarks} points, got {len(data)}"
)

rows = {frame: {"frame": frame} for frame in range(n_frames)}
for i, (frame, y, x) in enumerate(data):
    landmark = LANDMARKS[i // n_frames]   # which block of n_frames clicks this belongs to
    rows[int(frame)][f"{landmark}_x"] = x
    rows[int(frame)][f"{landmark}_y"] = y

df = pd.DataFrame(rows.values()).sort_values("frame").reset_index(drop=True)

## 7. Set up the output folder

Creates `landmarks_outputs/` inside `subject_data_vid_path` if it doesn't already exist.
All plots from here on are saved there; the raw coordinate CSV is saved one level up.

In [ ]:
# if it doesnt exist make a directory in the subject to save the csv and plots
output_dir = os.path.join(subject_data_vid_path, "landmarks_outputs")
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

In [ ]:
df.to_csv(os.path.join(subject_data_vid_path, "landmarks.csv"), index=False)

## 8. QC: overlay landmarks on each frame

Saves one PNG per frame (`landmarks_outputs/P{n}.png`) with the clicked landmarks and
labels drawn on top — use these to visually confirm nothing was mislabeled in step 6.

**Note the frame naming:** files are saved as `P{i+8}` (frame index + 8), which assumes
a specific numbering convention for this subject/session (e.g. video 0 corresponds to
session "P8"). **Check and adjust this offset** if numbering differs for a new
subject/session, otherwise the saved filenames will be misleading.

In [ ]:
# now visualise it based on the data in the table:
for i in range(len(frames)):
    plt.imshow(frames[i])
    plt.scatter(df.loc[i, [f"{lm}_x" for lm in LANDMARKS]], df.loc[i, [f"{lm}_y" for lm in LANDMARKS]], c='C0', s=10)
    # add text
    for lm in LANDMARKS:
        plt.text(df.loc[i, f"{lm}_x"]+10, df.loc[i, f"{lm}_y"]+10, lm, color='C0', fontsize=8)
    plt.title(f"P{i+8}")
    plt.axis('off')
    plt.savefig(os.path.join(output_dir, f"P{i+8}.png"), dpi=300, bbox_inches='tight')
    plt.show()

## 9. Pairwise distances (pixels)

Computes the Euclidean distance between every pair of landmarks, for every frame, in
raw pixel units, and saves three views:
- **absolute** — raw pixel distance per frame
- **relative** — change from frame 0 (baseline-subtracted)
- **percentage** — value as a % of frame 0

Pixel distances are **not** comparable across sessions/subjects if the camera
distance/angle/zoom changes — see the calibration step below for a corrected version.

In [ ]:
from itertools import combinations
import matplotlib.pyplot as plt
import numpy as np

pairs = list(combinations(LANDMARKS, 2))

dist_df = pd.DataFrame({"frame": df["frame"]})
for a, b in pairs:
    dx = df[f"{a}_x"] - df[f"{b}_x"]
    dy = df[f"{a}_y"] - df[f"{b}_y"]
    dist_df[f"{a}-{b}"] = np.sqrt(dx**2 + dy**2)

pair_cols = dist_df.columns[1:]

colors = plt.rcParams['axes.prop_cycle'].by_key()['color']  # 10 default colors
markers = ['o', 's', '^', 'D', 'v', 'P', 'X', '*', 'h', '<', '>']

def plot_lines(ax, x, plot_df):
    for i, col in enumerate(pair_cols):
        color = colors[i % len(colors)]
        marker = markers[i // len(colors)]
        ax.plot(x, plot_df[col], color=color, marker=marker, label=col)

# 1. absolute distance
fig, ax = plt.subplots(figsize=(4, 6))
plot_lines(ax, dist_df["frame"], dist_df)
ax.set_xlabel("frame"); ax.set_ylabel("distance (px)")
ax.set_title("Pairwise landmark distances across frames")
# ax.legend(bbox_to_anchor=(1.05, 1), loc="upper left", fontsize=8)
plt.tight_layout()
plt.savefig(os.path.join(output_dir, "pairwise_distances_absolute.png"), dpi=300, bbox_inches='tight')
plt.show()

# 2. relative (baseline-subtracted)
rel_df = dist_df.copy()
rel_df[pair_cols] = dist_df[pair_cols] - dist_df[pair_cols].iloc[0]

fig, ax = plt.subplots(figsize=(4, 6))
plot_lines(ax, rel_df["frame"], rel_df)
ax.axhline(0, color="gray", lw=0.8, ls="--")
ax.set_xlabel("frame"); ax.set_ylabel("distance change from frame 0 (px)")
ax.set_title("Relative pairwise distance (baseline-subtracted)")
# ax.legend(bbox_to_anchor=(1.05, 1), loc="upper left", fontsize=8)
plt.tight_layout()
plt.savefig(os.path.join(output_dir, "pairwise_distances_relative.png"), dpi=300, bbox_inches='tight')
plt.show()

# 3. percentage of frame 0
pct_df = dist_df.copy()
pct_df[pair_cols] = dist_df[pair_cols] / dist_df[pair_cols].iloc[0] * 100

fig, ax = plt.subplots(figsize=(4, 6))
plot_lines(ax, pct_df["frame"], pct_df)
ax.axhline(100, color="gray", lw=0.8, ls="--")
ax.set_xlabel("frame"); ax.set_ylabel("% of frame 0 distance")
ax.set_title("Relative pairwise distance (% of frame 0)")
# ax.legend(bbox_to_anchor=(1.05, 1), loc="upper left", fontsize=8)
plt.tight_layout()
plt.savefig(os.path.join(output_dir, "pairwise_distances_percentage.png"), dpi=300, bbox_inches='tight')
plt.show()

## 10. Calibrated distances (millimetres)

Converts pixel distances to millimetres using the distance between `headplate_a` and
`headplate_p` as a known physical reference (`REF_MM`, currently **5.0 mm** — this is the
true, fixed distance between those two points on the headplate hardware).

A separate pixel-to-mm scale factor is computed **per frame**, which corrects for
frame-to-frame differences in camera distance/angle/zoom — this is what makes the
calibrated plots comparable across sessions, unlike the raw pixel plots above.

**If the headplate design or `REF_MM` changes, update `REF_MM` here.** Also confirm
`REF_PAIR` still refers to two landmarks that are genuinely rigid/fixed relative to each
other — if you change which landmarks are tracked, the calibration reference needs to
stay a physically constant distance, otherwise the mm values will be wrong.

In [ ]:
# --- calibrate to real-world units using headplate_a-headplate_p as a 5 mm reference ---
REF_PAIR = "headplate_a-headplate_p"
REF_MM = 5.0

scale_mm_per_px = REF_MM / dist_df[REF_PAIR]   # one scale factor per frame, corrects for
                                                # camera distance/angle changes across sessions

mm_df = dist_df.copy()
mm_df[pair_cols] = dist_df[pair_cols].multiply(scale_mm_per_px, axis=0)

# 4. absolute distance, calibrated (mm)
fig, ax = plt.subplots(figsize=(3, 3))
plot_lines(ax, mm_df["frame"], mm_df)
ax.set_xlabel("frame"); ax.set_ylabel("distance (mm)")
ax.set_title("Pairwise landmark distances across frames (calibrated)")
ax.legend(bbox_to_anchor=(1.05, 1), loc="upper left", fontsize=8)
plt.tight_layout()
plt.savefig(os.path.join(output_dir, "pairwise_distances_legen.png"), dpi=300, bbox_inches='tight')
plt.show()

# 4. absolute distance, calibrated (mm)
fig, ax = plt.subplots(figsize=(4, 6))
plot_lines(ax, mm_df["frame"], mm_df)
ax.set_xlabel("frame"); ax.set_ylabel("distance (mm)")
ax.set_title("Pairwise landmark distances across frames (calibrated)")
# ax.legend(bbox_to_anchor=(1.05, 1), loc="upper left", fontsize=8)
plt.tight_layout()
plt.savefig(os.path.join(output_dir, "pairwise_distances_absolute_calibrated.png"), dpi=300, bbox_inches='tight')
plt.show()

# 5. relative (baseline-subtracted), calibrated (mm)
rel_mm_df = mm_df.copy()
rel_mm_df[pair_cols] = mm_df[pair_cols] - mm_df[pair_cols].iloc[0]

fig, ax = plt.subplots(figsize=(4, 6))
plot_lines(ax, rel_mm_df["frame"], rel_mm_df)
ax.axhline(0, color="gray", lw=0.8, ls="--")
ax.set_xlabel("frame"); ax.set_ylabel("distance change from frame 0 (mm)")
ax.set_title("Relative pairwise distance, calibrated (baseline-subtracted)")
plt.tight_layout()
plt.savefig(os.path.join(output_dir, "pairwise_distances_relative_calibrated.png"), dpi=300, bbox_inches='tight')
plt.show()

# 6. percentage of frame 0, calibrated
pct_mm_df = mm_df.copy()
pct_mm_df[pair_cols] = mm_df[pair_cols] / mm_df[pair_cols].iloc[0] * 100

fig, ax = plt.subplots(figsize=(4, 6))
plot_lines(ax, pct_mm_df["frame"], pct_mm_df)
ax.axhline(100, color="gray", lw=0.8, ls="--")
ax.set_xlabel("frame"); ax.set_ylabel("% of frame 0 distance (calibrated)")
ax.set_title("Relative pairwise distance, calibrated (% of frame 0)")
plt.tight_layout()
plt.savefig(os.path.join(output_dir, "pairwise_distances_percentage_calibrated.png"), dpi=300, bbox_inches='tight')
plt.show()